# Optional notebook - Clustering, and the limits of finding groups

**Outside the twelve sessions** · about 2 hours, self-paced

---

## Learning objectives

By the end of this notebook you will be able to:

1. State what question clustering answers, and what it cannot answer. `[CLO1]`
2. Run k-means, agglomerative clustering and DBSCAN, and say what assumption
   each one makes about the shape of a group. `[CLO6]`
3. Choose k with more than one index, and act correctly when the indices
   disagree. `[CLO5]`
4. Measure agreement between two clusterings, and between a clustering and a
   known label, without confusing the two. `[CLO5]`
5. Explain why a clustering result is a hypothesis rather than a finding.
   `[CLO2]`

## Prerequisites

Sessions 2, 3 and 6. You will need the split, the scaling discipline and the
habit of asking whether a difference is larger than the noise.

## Why this notebook exists

Every model in the twelve sessions is **supervised**: there is a target, and
being wrong is measurable. Clustering has no target. Nothing tells you the
answer is wrong, and that is exactly what makes it dangerous.

You will meet clustering as "customer segmentation", "patient phenotyping" or
"market segments". In each case somebody runs an algorithm, gets four groups,
names them, and starts making decisions. This notebook is about the question
that should come between the algorithm and the names: **is there anything
there at all?**

On our own data the answer will turn out to be "not much", and reporting that
is the skill being taught.

## §1 - The question clustering answers

Supervised learning asks: **given these features, what is the value of that
target?** There is a right answer, held out, and you can be scored against it.

Clustering asks something much weaker: **do these rows fall into groups that
are more similar within than between?** There is no right answer held out.
Change the features, the scaling or the algorithm and you get different
groups, and none of them is *wrong*.

Three consequences follow, and they are the whole notebook:

1. **You choose the geometry.** "Similar" means a distance, a distance means a
   scale, and a scale is your decision. The clusters are partly your artifact.
2. **The algorithm always returns something.** Ask k-means for four groups in
   pure noise and you will receive four groups. Nothing raises.
3. **A cluster is a hypothesis.** "These 900 listings are similar on five
   measurements" is a finding. "These are the budget family segment" is a
   story you added afterwards, and it needs its own evidence.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# scipy owns the dendrogram; sklearn has no equivalent.
from scipy.cluster.hierarchy import cophenet, dendrogram, linkage
from scipy.spatial.distance import pdist

from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_moons
# Two families of index. The first three need true labels, the last two do not,
# and §7 is about never confusing the two situations.
from sklearn.metrics import (adjusted_rand_score, completeness_score,
                             davies_bouldin_score, homogeneity_score,
                             silhouette_score, v_measure_score)
from sklearn.preprocessing import StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
set_seed(SEED)

# The same split and the same scoping as Sessions 6 onwards. Clustering has no
# target to leak, but the sealed test set stays sealed: the rule does not have
# exceptions for techniques that seem harmless.
df = load_raw()
train, _ = split_by_host(df)
train["price_num"] = (
    train["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan).astype(float)
)
work = train[train["price_num"].notna()]
work = work[work["price_num"].between(10, 2000)]
work = work[work["minimum_nights"] <= 3].copy()

FEATURES = ["accommodates", "bedrooms", "bathrooms", "latitude", "longitude"]
# dropna() rather than imputation, and deliberately: an imputed value sits at the
# centre of the distribution, which pulls a listing towards whichever cluster
# happens to live there. Imputation and distance-based methods interact badly.
sub = work[FEATURES + ["price_num", "neighbourhood_group_cleansed",
                       "room_type"]].dropna()

X_raw = sub[FEATURES].to_numpy(dtype=float)
X = StandardScaler().fit_transform(X_raw)

print(f"listings after scoping : {len(work):,}")
print(f"complete on {len(FEATURES)} features: {len(sub):,}")
print(f"districts represented  : {sub['neighbourhood_group_cleansed'].nunique()}")

## §2 - k-means, and the decision you make before you run it

k-means is three lines of idea:

1. Put $k$ centres somewhere.
2. Assign every point to its nearest centre.
3. Move each centre to the mean of its points. Repeat until nothing moves.

It minimises the **inertia**, the sum of squared distances from each point to
its own centre. That objective is why k-means behaves as it does: squared
distance, so it is drawn to compact round blobs of roughly equal size, and
*nearest* centre, so the boundaries are straight lines.

### The decision that comes first

"Nearest" is a distance, and a distance is dominated by whichever feature has
the largest numbers. Look at the spread of our five features before running
anything.

### The algorithm, in three pictures

![kmeans steps](../assets/diagrams/clustering/kmeans-steps.png)

Assign every point to the nearest centre; move each centre to the mean of the points assigned to it; repeat until nothing moves.

Now ask what **nearest** means. It is a distance, a distance has a scale, and the scale is a decision that nobody in this picture made for you. That question is the rest of this section.

In [ ]:
# Standard deviations on the raw scale. Capacity varies by a couple of units;
# latitude varies by about a hundredth of a degree. In a Euclidean distance the
# geography contributes essentially nothing.
spread = pd.Series(X_raw.std(axis=0), index=FEATURES, name="raw sd")
print(spread.round(4).to_string())
print(f"\nlargest / smallest = {spread.max() / spread.min():,.0f}x")

In [ ]:
# TODO: Cluster the listings twice with k=4, once on X_raw and once on X.
#
#   1. Fit KMeans(n_clusters=4, n_init=10, random_state=SEED) on each.
#   2. Measure how much the two labellings agree, with adjusted_rand_score.
#   3. For each version, print the mean of every raw feature per cluster.
#
# Then answer in a comment: which feature is the raw version actually
# clustering on, and how can you tell from the per-cluster means?

## §3 - Choosing k, and what to do when the indices disagree

k-means needs $k$ and will not suggest one. Three common ways to choose it:

| method | what it measures | how it fails |
|---|---|---|
| **elbow** | inertia against k, looking for a bend | inertia always falls, so there is often no bend |
| **silhouette** | how much closer a point is to its own cluster than to the next | favours few, well separated clusters |
| **Davies-Bouldin** | within-cluster spread over between-cluster distance, lower is better | favours compact clusters of similar size |

Run all three. The interesting case is when they disagree, and on our data
they will.

In [ ]:
ks = range(2, 11)
rows = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    rows.append({
        "k": k,
        # Inertia falls monotonically by construction: more centres always means
        # shorter distances. On its own it can never choose k.
        "inertia": km.inertia_,
        # sample_size keeps the silhouette affordable - it is O(n^2) in full, and
        # 5,000 points would mean 25 million pairwise distances per value of k.
        "silhouette": silhouette_score(X, km.labels_, sample_size=3000,
                                       random_state=SEED),
        "davies_bouldin": davies_bouldin_score(X, km.labels_),
        # The size of the smallest cluster, which no index reports and which often
        # settles the decision on its own: a "segment" of 40 listings is not one.
        "smallest": int(np.bincount(km.labels_).min()),
    })

sweep = pd.DataFrame(rows).set_index("k")
print(sweep.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
# Left: the elbow that is not there. A smooth curve with no bend is the normal
# case, and treating a gentle knee as an answer is how k gets chosen by wishful
# thinking.
axes[0].plot(sweep.index, sweep["inertia"], "o-", color="steelblue")
axes[0].set(title="elbow: look for a bend (there is none)", xlabel="k",
            ylabel="inertia")
# Middle: higher is better.
axes[1].plot(sweep.index, sweep["silhouette"], "o-", color="darkorange")
axes[1].set(title="silhouette: higher is better", xlabel="k")
# Right: lower is better. Note the two panels do not agree on a winner.
axes[2].plot(sweep.index, sweep["davies_bouldin"], "o-", color="seagreen")
axes[2].set(title="Davies-Bouldin: lower is better", xlabel="k")
plt.tight_layout()
plt.show()

best_sil = int(sweep["silhouette"].idxmax())
best_db = int(sweep["davies_bouldin"].idxmin())
print(f"silhouette prefers     k = {best_sil}  ({sweep['silhouette'].max():.4f})")
print(f"Davies-Bouldin prefers k = {best_db}  ({sweep['davies_bouldin'].min():.4f})")

### The number that matters more than the winner

A silhouette runs from $-1$ to $+1$. Our best value is about **0.33**.

As a rough reading: above 0.7 is strong structure, 0.5 to 0.7 is reasonable,
0.25 to 0.5 is weak, and below 0.25 means no substantial structure was found.
We are at the bottom of "weak", at the *best* value of $k$ we tried.

So the honest summary of this section is not "we found four segments". It is:

> The listings do not separate into well-defined groups on these five
> features. Any partition we impose will be a convenience, not a discovery,
> and the two indices we used do not even agree on how many pieces to cut.

That sentence is worth more to a client than a coloured scatter plot with
four names on it.

## §4 - Where k-means fails, and why the failures are structural

k-means minimises squared distance to a centre. That single fact predicts
every one of its failures. Three cases, all synthetic so the truth is known.

In [ ]:
# TODO: Predict before running.
#
# For each of these three datasets, which algorithm will recover the true
# groups - k-means, agglomerative (ward), or DBSCAN?
#
#   unequal variance : ____
#   anisotropic      : ____
#   two moons        : ____
#
# Then build the table and measure it with adjusted_rand_score against the
# generated labels.

## §5 - Agglomerative clustering, and reading a dendrogram

Agglomerative clustering starts with every point as its own cluster and
repeatedly merges the two closest, until one cluster remains. It never needs
$k$ in advance: it produces the whole nested hierarchy, and you cut it where
you like.

"Closest" needs a definition for *clusters* rather than points, and that is
the **linkage**:

| linkage | merges the pair with the smallest | tends to produce |
|---|---|---|
| `single` | distance between their nearest members | long straggly chains |
| `complete` | distance between their furthest members | compact, equal-sized blobs |
| `average` | mean distance over all cross pairs | a compromise |
| `ward` | increase in total within-cluster variance | blobs, much like k-means |

The linkage is the same kind of choice as the scaler: it decides the answer.

### Linkage, and the tree it produces

![cluster distances](../assets/diagrams/clustering/cluster-distances.png)

![hierarchical clusters](../assets/diagrams/clustering/hierarchical-clusters.png)

The first shows the four linkage rules as different definitions of the distance between two *groups*: nearest members, furthest members, average over all pairs, and increase in total variance. The second shows a dendrogram beside the clusters it encodes.

Linkage is the same kind of choice as the scaler - it decides the answer rather than merely tidying up. To read the dendrogram: height is the distance at which a merge happened, so a tall vertical line means two dissimilar groups were joined, and just below such a line is where a cut makes sense.

In [ ]:
# A deliberately tiny dataset. A dendrogram over 5,000 listings is a black
# smear; over twelve points every merge is visible.
X_small, _ = make_blobs(n_samples=12, centers=3, random_state=1)

# scipy's linkage returns the merge history: one row per merge, holding
# [cluster a, cluster b, distance at which they merged, size of the result].
Z = linkage(X_small, method="ward")
merges = pd.DataFrame(Z, columns=["a", "b", "distance", "size"])
print(merges.round(3).to_string())
# The jump in the distance column is the signal. A merge that costs far more
# than the previous one is joining two things that were not close, which is a
# hint about where to cut. np.diff gives the change from each merge to the next,
# so entry j of it sits between merge j and merge j+1.
jump = int(np.argmax(np.diff(Z[:, 2])))
print(f"\nlargest jump in merge distance: between merge {jump} "
      f"({Z[jump, 2]:.3f}) and merge {jump + 1} ({Z[jump + 1, 2]:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# Left: every merge. Height is the distance at which the merge happened, so a
# tall vertical line means two dissimilar groups were joined.
dendrogram(Z, ax=axes[0])
axes[0].set(title="every merge, 12 points", xlabel="point", ylabel="merge distance")

# Right: a larger set, truncated. truncate_mode="lastp" with p=12 shows only the
# final 12 merges and collapses the rest, which is the only way a dendrogram
# stays readable beyond a few dozen points.
X_big, _ = make_blobs(n_samples=150, centers=3, random_state=1)
Z_big = linkage(X_big, method="ward")
dendrogram(Z_big, ax=axes[1], truncate_mode="lastp", p=12,
           show_leaf_counts=True)
axes[1].set(title="150 points, truncated to the last 12 merges",
            xlabel="cluster (n in brackets)")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Compare the four linkages on X_big.
#
#   1. For each of ward, complete, average and single, build the linkage matrix.
#   2. Compute the cophenetic correlation with scipy.cluster.hierarchy.cophenet
#      against pdist(X_big).
#   3. Say in one sentence what that number measures, and what a high value does
#      NOT prove.

## §6 - DBSCAN, and a parameter that looks innocent

DBSCAN takes a different view: a cluster is a **dense region**. It needs two
settings.

- `eps` - the radius that counts as "nearby".
- `min_samples` - how many neighbours within that radius make a point a
  **core** point.

Every point is then one of three things: a **core** point, a **border** point
(inside a core point's radius but not dense itself), or **noise** (neither).
Noise is labelled `-1` and belongs to no cluster.

Two properties follow, and they are why DBSCAN is worth knowing:

- It finds clusters of **any shape**, because it grows them by connection
  rather than by distance to a centre. That is the two-moons result in §4.
- It has a category for "this point is not in any group". k-means does not:
  every point is assigned, including the ones that plainly belong nowhere.

It does **not** ask for $k$. This is usually sold as the headline advantage,
and it is half true.

### Core, border and noise

![dbscan](../assets/diagrams/clustering/dbscan.png)

Every point is one of three things: a **core** point with enough neighbours inside its radius, a **border** point that is inside someone else's radius but not dense itself, or **noise** belonging to no cluster at all.

That third category is the real difference from k-means, which has no way of saying *this point is not in any group* - every point is assigned to something, however badly it fits. Whether that category is a feature or a failure depends on the sweep below, where at the tightest radius half our listings become noise.

In [ ]:
# Geography only, scaled, so that eps is a distance on a comparable plane.
geo = StandardScaler().fit_transform(sub[["latitude", "longitude"]])

rows = []
for eps in (0.05, 0.10, 0.15, 0.20, 0.30):
    labels = DBSCAN(eps=eps, min_samples=10).fit_predict(geo)
    # -1 is the noise label and is not a cluster, so it is excluded from the count.
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    rows.append({"eps": eps, "clusters": n_clusters,
                 "noise_%": (labels == -1).mean() * 100,
                 "largest_cluster": int(np.bincount(labels[labels >= 0]).max())
                 if n_clusters else 0})
print(pd.DataFrame(rows).round(2).to_string(index=False))

print("""
DBSCAN does not ask for k, and then eps decides it anyway: 64 clusters at
eps=0.05, two at eps=0.30. The parameter moved from "how many groups" to "how
close is close", which is harder to reason about and easier to tune until the
picture looks right.

Note the noise column as well. At eps=0.05 half the listings belong to no
cluster at all. Whether that is a feature or a failure depends on whether you
believe half the market really is unlike everything else.
""")

## §7 - Evaluating a clustering

There are two completely different situations, and the commonest mistake in
this whole area is using a tool from one in the other.

### When true labels ARE known

This is not the real case. It happens in teaching, and when you are checking
whether a clustering recovers something you already understand.

- **Adjusted Rand Index** - agreement between two partitions, corrected for
  chance. 1.0 identical, 0.0 chance, negative worse than chance.
- **v-measure** - the harmonic mean of *homogeneity* (each cluster contains
  one class) and *completeness* (each class sits in one cluster).

Both ignore the label names, which is what makes them usable at all: cluster
"2" and class "cat" cannot be compared directly.

### When true labels are NOT known

This is the real case, and the indices are much weaker.

- **Silhouette** - per point, how much closer it is to its own cluster than to
  the nearest other one.
- **Davies-Bouldin** - within-cluster spread relative to between-cluster
  separation, lower better.

Both measure *whether the partition is geometrically tidy*. Neither can tell
you the groups mean anything.

In [ ]:
# The behaviour of ARI and v-measure on four small cases, so their failure modes
# are concrete rather than described.
cases = [
    ("identical up to renaming", [0, 0, 1, 1], [1, 1, 0, 0]),
    ("one cluster split in two", [0, 0, 1, 1], [0, 1, 2, 2]),
    ("everything in one cluster", [0, 0, 1, 1], [0, 0, 0, 0]),
    ("every point its own cluster", [0, 0, 1, 1], [0, 1, 2, 3]),
]
print(f"{'case':30s} {'ARI':>7s} {'v-meas':>7s} {'homog':>7s} {'complete':>9s}")
for name, truth, pred in cases:
    print(f"{name:30s} {adjusted_rand_score(truth, pred):>7.3f} "
          f"{v_measure_score(truth, pred):>7.3f} "
          f"{homogeneity_score(truth, pred):>7.3f} "
          f"{completeness_score(truth, pred):>9.3f}")

print("""
The last two rows are the ones to study. Putting every point in its own cluster
gives perfect homogeneity - each cluster really does contain a single class -
and it is a useless clustering. Homogeneity alone can always be maximised by
making more clusters, which is why it is never reported on its own.
""")

In [ ]:
# TODO: Do the clusters correspond to the districts?
#
#   1. Cluster `geo` (latitude and longitude, scaled) with k=10 - the city has
#      exactly 10 districts.
#   2. Compare the labels against sub["neighbourhood_group_cleansed"] with ARI
#      and v-measure.
#   3. Then cluster the full five-feature X with k=10 and compare that too.
#
# Write one sentence on what a middling ARI tells you, and one on what it does
# not tell you.

## §8 - The close

Everything measured in this notebook points one way.

| what we asked | what we got |
|---|---|
| how strong is the structure? | silhouette 0.33 at best, which is weak |
| how many clusters? | silhouette says 2, Davies-Bouldin says 4 |
| does scaling matter? | ARI 0.28 between raw and scaled. It decides the answer |
| is one algorithm best? | no. DBSCAN scores 1.00 and 0.00 on different shapes |
| do the clusters match the districts? | partly, and the mismatch is not an error |

So the reportable result is that **the Barcelona listings do not fall into
well-separated groups on these features**. You could still cut them into four
pieces and give the pieces names, and the pieces would be no less real for
being weak. But the sentence that has to go next to them is "these are a
convenient partition of a continuum, not discovered types".

### Three questions to ask anyone who shows you a segmentation

1. **What were the features, and how were they scaled?** If they cannot
   answer, the segments are an artifact of whichever column had the largest
   numbers.
2. **How strong is the structure?** Ask for the silhouette, or any measure of
   separation. "We found four segments" is not an answer to this.
3. **What changes if there are three, or five?** If the recommendation
   survives a different $k$, the clustering was not really carrying it. If it
   does not survive, the recommendation rests on a choice nobody defended.

That is the transferable part. You will be shown segmentations far more often
than you will build them.

## §9 - Knowledge check

From memory, and then check.

1. Why does k-means fail on two interleaved crescents, and why does no choice
   of $k$ fix it?
2. You scale your features and the clusters change completely. Which
   clustering is correct?
3. A colleague reports "silhouette 0.31, maximised at k=6, so we have six
   customer segments". Name two things wrong with that sentence.
4. What does a DBSCAN label of `-1` mean, and what is the equivalent label in
   k-means?
5. Homogeneity is 1.00. Is the clustering good?